In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.pipeline import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
def train_churn_model(table_name = "nff_catalog_f586_dev.gold_ml.churn_features"):
    pipeline_stages = []

    #TASK 1: CREATE A DATAFRAME
    df = spark.read.table(table_name)
    
    #TASK 2: STRINGS TO NUMBERS
    categorical_col = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, StringType) and f.name != "is_churned"
    ]
    for col in categorical_col:
        string_indexer = StringIndexer(inputCol = col, outputCol = col + "_index", handleInvalid="keep")
        encoder = OneHotEncoder(inputCol=string_indexer.getOutputCol(), outputCol=col + "_vec", dropLast=False)
        pipeline_stages += [string_indexer, encoder]
   
    #TASK 3: NUMERICAL FEATURES TO VectorAssembler
    numeric_types = (IntegerType, LongType, FloatType, DoubleType, DecimalType)
    numerical_cols = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, numeric_types) and f.name != "is_churned"
    ]
    feature_cols = numerical_cols + [c + "_vec" for c in categorical_col]
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
    pipeline_stages += [assembler]
    
    #TASK 4: NORMALIZE THE "outputCol" OF VectorAssembler USING StandardScaler
    scaler = StandardScaler(inputCol='raw_features', outputCol='features')
    pipeline_stages += [scaler]

    #TASK 5: DEFINE THE MODEL
    rf = RandomForestClassifier(labelCol="is_churned", featuresCol="features", numTrees=10)
    pipeline_stages += [rf]
 
    #TASK 6: BUILD THE PIPELINE
    pipeline = Pipeline(stages=pipeline_stages) 
    
    #TASK 7: SPLIT DATAFRAME
    train_data, test_data = df.randomSplit([0.8,0.2], seed=42)
    
    #TASK 8: TRAIN
    print("🚀 Training the Random Forest model...")
    model = pipeline.fit(train_data)
    print("✅ Done!")
   
    #TASK 9: PREDICTION
    predictions = model.transform(test_data)
    display(predictions)

    #TASK 10: EVALUATE
    evaluator = BinaryClassificationEvaluator(labelCol="is_churned", metricName="areaUnderROC")
    auc = evaluator.evaluate(predictions)
      
    print("-" * 30)
    print(f"✅ Model Training Complete!")
    print(f"📊 Area Under ROC (Accuracy): {auc:.4f}")
    print("-" * 30)

    # Show a sample of results
    predictions.select("customer_id", "probability", "prediction", "is_churned").show(10)
    
    return model
    

In [0]:
 model = train_churn_model()